# Stages 2 + 3 on Colab - agent search, then the one-time test evaluation

Single T4 (15 GB): the search policy uses **Qwen2.5-3B-Instruct** in fp16 (the 7B model
needs the two T4s Kaggle provides). Everything else is identical to the Kaggle notebook.

In [ ]:
# Colab: features and results live on Google Drive so they survive the runtime.
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, subprocess
WORK = "/content/drive/MyDrive/pathfusion"
os.makedirs(WORK, exist_ok=True)
REPO_DIR = "/content/pathology-fusion-AI"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", "https://github.com/jkaral/pathology-fusion-AI.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements-kaggle.txt
for name in ("HF_TOKEN", "WANDB_API_KEY"):   # Colab: key icon in the left sidebar -> add secret
    try:
        os.environ[name] = userdata.get(name); print(name, "loaded")
    except Exception:
        print(name, "not set")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
EMB = f"{WORK}/embeddings"
OVR = f"paths.agent_dir={WORK}/agent paths.results_dir={WORK}/results agent.llm.local_model=Qwen/Qwen2.5-3B-Instruct"
!python scripts/02_run_agent.py --embeddings-dir {EMB} {OVR}

In [ ]:
!python scripts/03_final_evaluation.py --embeddings-dir {EMB} {OVR}
!python scripts/04_report_facts.py --embeddings-dir {EMB} {OVR}